# Training Log 분석 (Jupyter 서버)

MoCo v2 / MoCo v3 학습 로그를 파싱해서 시각화.

- **Cell 1**: 작업 디렉토리(레포 루트) + 로그 파싱
- **Cell 2**: 그래프 출력 (Loss / feat_std / LR)
- **Cell 3**: 요약 수치

In [ ]:
# Cell 1 — 작업 디렉토리 + 로그 파싱
import re
from pathlib import Path

import os
from pathlib import Path
_ROOT = Path.cwd()
if _ROOT.name == 'notebooks':
    os.chdir(_ROOT.parent)

LOG_FILES = {
    'mocov2':       'logs/mocov2_seed42.log',
    'mocov3_vits':  'logs/mocov3_vits_seed42.log',
    'mocov3_vits8': 'logs/mocov3_vits8_seed42.log',   # ★ 최종 모델
}

def parse_log(path):
    """로그 파일 파싱. epoch 요약과 step 로그 모두 추출."""
    epoch_re = re.compile(
        r'\[Ep\s*(\d+)\s*done\]\s*avg_loss=([\d.]+)\s*\|\s*avg_feat_std=([\d.]+)\s*\|\s*time=([\d.]+)s'
    )
    step_re = re.compile(
        r'Ep\s*(\d+)\s*\|\s*step\s*(\d+)/\d+\s*\|\s*loss\s*([\d.]+)\s*\|\s*feat_std\s*([\d.]+)\s*\|\s*lr\s*([\d.]+)'
    )

    epochs, losses, feat_stds, times = [], [], [], []
    step_epochs, step_lrs = [], []

    with open(path, encoding='utf-8') as f:
        for line in f:
            m = epoch_re.search(line)
            if m:
                epochs.append(int(m.group(1)))
                losses.append(float(m.group(2)))
                feat_stds.append(float(m.group(3)))
                times.append(float(m.group(4)))
                continue
            m = step_re.search(line)
            if m:
                ep   = int(m.group(1))
                step = int(m.group(2))
                lr   = float(m.group(5))
                step_epochs.append(ep + step / 390)
                step_lrs.append(lr)

    return {
        'epochs':      epochs,
        'losses':      losses,
        'feat_stds':   feat_stds,
        'times':       times,
        'step_epochs': step_epochs,
        'step_lrs':    step_lrs,
    }

data = {}
for name, path in LOG_FILES.items():
    if Path(path).exists():
        data[name] = parse_log(path)
        ep_count = len(data[name]['epochs'])
        last_loss = data[name]['losses'][-1] if ep_count else 'N/A'
        print(f'[{name}] {ep_count} epochs parsed, last loss={last_loss}')
    else:
        print(f'[{name}] 로그 파일 없음: {path}')

In [ ]:
# Cell 2 — 그래프
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

COLORS = {'mocov2': '#2196F3', 'mocov3_vits': '#FF9800', 'mocov3_vits8': '#FF5722'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('SSL Pretraining Progress', fontsize=13, fontweight='bold')

ax_loss, ax_std, ax_lr = axes

for name, d in data.items():
    c = COLORS.get(name, 'gray')
    ep = d['epochs']
    if not ep:
        continue

    # --- Loss ---
    ax_loss.plot(ep, d['losses'], color=c, linewidth=1.5, label=name)

    # --- feat_std ---
    ax_std.plot(ep, d['feat_stds'], color=c, linewidth=1.5, label=name)

    # --- LR ---
    if d['step_lrs']:
        ax_lr.plot(d['step_epochs'], d['step_lrs'], color=c, linewidth=1.0,
                   alpha=0.8, label=name)

# collapse 경계선
ax_std.axhline(0.05, color='red', linestyle='--', linewidth=1.0,
               label='collapse threshold (0.05)')

# 축 설정
for ax, title, ylabel in [
    (ax_loss, 'Loss (per epoch)', 'avg_loss'),
    (ax_std,  'Feature Std (per epoch)', 'avg_feat_std'),
    (ax_lr,   'Learning Rate (per step)', 'lr'),
]:
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(f'{DRIVE_DIR}/logs/training_progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장: logs/training_progress.png')

In [ ]:
# Cell 3 — 요약 수치
for name, d in data.items():
    ep = d['epochs']
    if not ep:
        continue
    total_h = sum(d['times']) / 3600
    avg_ep  = sum(d['times']) / len(d['times'])
    print(f'=== {name} ===')
    print(f'  완료 epoch     : {ep[0]} → {ep[-1]}  ({len(ep)} epochs)')
    print(f'  loss           : {d["losses"][0]:.4f} → {d["losses"][-1]:.4f}')
    print(f'  feat_std       : {d["feat_stds"][0]:.4f} → {d["feat_stds"][-1]:.4f}')
    print(f'  epoch당 소요   : {avg_ep:.0f}s  ({avg_ep/60:.1f}min)')
    print(f'  총 소요 시간   : {total_h:.1f}h')
    if d['step_lrs']:
        print(f'  현재 lr        : {d["step_lrs"][-1]:.5f}')
    print()